In [ ]:
import pandas as pd
import requests
import os
from io import StringIO

# Definiere den Pfad zum Raw-Ordner relativ zum Notebook
RAW_DIR = "../data/01_raw/"

# Erstelle den Ordner, falls er noch nicht existiert
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Zielordner bereit: {RAW_DIR}")

In [ ]:
import zipfile
import io
import time
import os
import requests
from dotenv import load_dotenv

# dotenv laden
load_dotenv('../.env')

# --- KONFIGURATION ---
TOKEN = os.getenv("GENESIS_API_TOKEN")
BASE_URL = 'https://www-genesis.destatis.de/genesisWS/rest/2020/'
HEADERS = {'Content-Type': 'application/x-www-form-urlencoded', 'username': TOKEN, 'password': ""}

def download_genesis_data(table_code, filename, startyear="1984", endyear="2024", force_reload=False):
    filepath = os.path.join(RAW_DIR, filename)
    
    # Caching: Überspringen, wenn Datei schon existiert
    if os.path.exists(filepath) and not force_reload:
        print(f"Überspringe Download, Datei existiert bereits: {filename}")
        return filepath

    print(f"Starte Download: {table_code} ({startyear}-{endyear})...")
    url = BASE_URL + 'data/tablefile'
    payload = {
        "name": table_code,
        "area": "all",
        "compress": "false",
        "transpose": "false",
        "startyear": str(startyear),
        "endyear": str(endyear),
        "language": "de",
        "format": "ffcsv",
        "job": "false"
    }
    
    try:
        r = requests.post(url, headers=HEADERS, data=payload)
        
        # Check ob Request okay und kein Fehler-JSON (Genesis schickt oft Status 200 mit Fehlertext)
        if r.status_code == 200 and "{\"Code\":" not in r.text[:100]:
            if r.content[:2] == b'PK':
                with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                    with z.open(z.namelist()[0]) as zf, open(filepath, 'wb') as f:
                        f.write(zf.read())
            else:
                with open(filepath, 'w', encoding='utf-8') as f:
                    f.write(r.text)
            print(f"   Gespeichert unter: {filepath}")
            return filepath
        else:
            print(f"   API-Fehler: {r.text[:200]}")
            return None
            
    except Exception as e:
        print(f"   Fehler beim Download: {e}")
        return None

In [ ]:
import pandas as pd
import glob

# Strafverfolgungsstatistik (Justiz) 24311-0002 -> Chunked Download
# API limitiert große Zeiträume, daher in Chunks aufteilen
YEAR_CHUNKS = [
    (1984, 1995),
    (1996, 2005),
    (2006, 2015),
    (2016, 2024)
]

print("=== Justizdaten Chunks ===")
justiz_files = []
for s, e in YEAR_CHUNKS:
    chunk_name = f"justiz_roh_{s}_{e}.csv"
    res = download_genesis_data("24311-0002", chunk_name, startyear=s, endyear=e)
    if res:
        justiz_files.append(res)
    time.sleep(1) # Kurze Pause, um die API nicht zu überlasten

# Alle Chunks mit Pandas laden und zusammenführen
print("\n=== Kombiniere Justizdaten ===")
df_list = []
for file in justiz_files:
    if file and os.path.exists(file):
        df_list.append(pd.read_csv(file, sep=";", encoding="utf-8", dtype=str))

if df_list:
    df_justiz = pd.concat(df_list, ignore_index=True)
    combined_path = os.path.join(RAW_DIR, "justiz_roh_kombiniert.csv")
    df_justiz.to_csv(combined_path, sep=";", index=False, encoding="utf-8")
    print(f"Kombinierte Datei gespeichert: {combined_path} ({len(df_justiz)} Zeilen)")

print("\n=== Bevölkerungsdaten ===")
# Bevölkerungsfortschreibung (klein genug für einen einzigen Request)
download_genesis_data("12411-0002", "bevoelkerung_roh.csv")

In [ ]:
# PKS Downloads: Aufteilung nach Nationalität (Deutsche / Nichtdeutsche Tatverdächtige)
# Hinweis: Die BKA-Links folgen meist diesem Namensschema (02 für deutsch, 03 für nichtdeutsch).
PKS_URLS = {
    "pks_deutsche_roh.xlsx": "https://www.bka.de/SharedDocs/Downloads/DE/Publikationen/PolizeilicheKriminalstatistik/2024/Interpretation/Tatverdaechtige/ZR-TV-04-T40-TV-insg-deutsch_xls.xlsx?__blob=publicationFile&v=3",
    "pks_nichtdeutsche_roh.xlsx": "https://www.bka.de/SharedDocs/Downloads/DE/Publikationen/PolizeilicheKriminalstatistik/2024/Interpretation/Tatverdaechtige/ZR-TV-07-T50-TV-insg-nichtdeutsch_xls.xlsx?__blob=publicationFile&v=3"
}

print("=== Starte Download der PKS Excel-Dateien ===")

for filename, url in PKS_URLS.items():
    filepath = os.path.join(RAW_DIR, filename)
    
    # Caching: Überspringen, wenn Datei schon existiert
    if os.path.exists(filepath):
        print(f"Überspringe Download, Datei existiert bereits: {filename}")
        continue
        
    print(f"Lade {filename} herunter...")
    try:
        response = requests.get(url)
        response.raise_for_status() # Prüft auf HTTP-Fehler
        
        with open(filepath, "wb") as f:
            f.write(response.content)
        print(f"   Gespeichert unter: {filepath}")
        
    except requests.exceptions.HTTPError as e:
        print(f"   HTTP-Fehler beim Herunterladen von {filename}: {e}")
        print(f"      Bitte Link prüfen, falls er sich beim BKA geändert hat: {url}")
    except Exception as e:
        print(f"   Fehler beim Herunterladen von {filename}: {e}")